# 4. Stage 4. ESCO Skills Extraction

**Documentation:** [Notebook guide](README.md) · [Stage data description](../../data/data-pipeline/stage_04/README_data.md)


**Stage:** 4 of 5

**Purpose:** Verify and enrich each vacancy's Stage 3 ESCO occupation assignment. The notebook applies exact and fuzzy title matching, followed by an ESCO-code fallback, and then adds the occupation's ESCO skills.

**Input:**
- Stage 3 result pickles referenced by the Stage 3 process tracker
- ESCO reference CSV files referenced by `g.Config.STAGE4_ESCO_DATA_PATH`

**Output:** One `ua-YYYY-MM-DD.pkl` file per completed day in `g.Config.STAGE4_OUTPUT_PATH`. Each output retains the Stage 3 fields and adds `esco_title`, `esco_id`, `esco_code`, `esco_skills`, `skill_labels_en`, and `extract_type`.

**Run:** Execute all cells from top to bottom. Existing Stage 4 output files are skipped. No API calls are required.

## 4.1. Load packages and configuration

Load the shared configuration from `code/data-pipeline/` and import the local libraries used for ESCO matching.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
import re
import sys
import time
from typing import Iterable, Optional

import pandas as pd
from rapidfuzz import fuzz, process

from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g

g.clean_memory()

## 4.2. Load ESCO reference data and Stage 3 tracker

Load the ESCO occupation, skill, and occupation-skill relation tables. Then load the Stage 3 process tracker; only rows with `result_status == 'complete'` are processed.

In [ ]:
occupation_skills_df = pd.read_csv(
    os.path.join(g.Config.STAGE4_ESCO_DATA_PATH, "occupationSkillRelations_en.csv")
)
occupations_df = pd.read_csv(
    os.path.join(g.Config.STAGE4_ESCO_DATA_PATH, "occupations_en.csv")
)
skills_df = pd.read_csv(
    os.path.join(g.Config.STAGE4_ESCO_DATA_PATH, "skills_en.csv")
)

process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
process_df = process_df.sort_values(by="input_file").reset_index(drop=True)

g.check_folder_exists(g.Config.STAGE4_OUTPUT_PATH)

## 4.3. Matching helpers

The title matcher normalises labels, retrieves a small RapidFuzz candidate set, and applies a deterministic combined similarity score. Short labels require a stricter effective threshold. The code matcher resolves exact codes, descendants, parents, and finally the nearest numeric ESCO root.

In [ ]:
def is_missing(value) -> bool:
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def normalize_text(text: str) -> str:
    if is_missing(text):
        return ""
    text = str(text).lower().strip()
    text = text.replace("/", " ").replace("-", " ")
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def singularize_token(token: str) -> str:
    if len(token) <= 3:
        return token
    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"
    if token.endswith("s") and not token.endswith("ss"):
        return token[:-1]
    return token


def normalize_label(text: str) -> str:
    return " ".join(singularize_token(token) for token in normalize_text(text).split())


def common_prefix_len(left: str, right: str) -> int:
    length = 0
    for left_char, right_char in zip(left, right):
        if left_char != right_char:
            break
        length += 1
    return length


def prefix_score(left: str, right: str) -> float:
    if not left or not right:
        return 0.0
    return common_prefix_len(left, right) / max(len(left), len(right))


def lcs_length(left: str, right: str) -> int:
    previous = [0] * (len(right) + 1)
    for left_char in left:
        current = [0]
        for index, right_char in enumerate(right, start=1):
            if left_char == right_char:
                current.append(previous[index - 1] + 1)
            else:
                current.append(max(previous[index], current[index - 1]))
        previous = current
    return previous[-1]


def lcs_score(left: str, right: str) -> float:
    if not left or not right:
        return 0.0
    return lcs_length(left, right) / max(len(left), len(right))


def char_ngrams(text: str, size: int = 3) -> set[str]:
    text = text.replace(" ", "")
    if len(text) < size:
        return {text} if text else set()
    return {text[index:index + size] for index in range(len(text) - size + 1)}


def ngram_jaccard(left: set[str], right: set[str]) -> float:
    if not left and not right:
        return 1.0
    if not left or not right:
        return 0.0
    return len(left & right) / len(left | right)


def score_pair(label: str, label_ngrams: set[str], choice: str, choice_ngrams: set[str]) -> float:
    prefix = prefix_score(label, choice)
    trigrams = ngram_jaccard(label_ngrams, choice_ngrams)
    lcs = lcs_score(label, choice) if max(len(label), len(choice)) <= 32 else 0.0
    label_tokens = label.split()
    choice_tokens = choice.split()

    score = (
        0.22 * fuzz.WRatio(label, choice) / 100.0
        + 0.16 * fuzz.ratio(label, choice) / 100.0
        + 0.10 * fuzz.partial_ratio(label, choice) / 100.0
        + 0.10 * fuzz.token_sort_ratio(label, choice) / 100.0
        + 0.08 * fuzz.token_set_ratio(label, choice) / 100.0
        + 0.14 * prefix
        + 0.10 * trigrams
        + 0.07 * lcs
        + 0.02 * float(label == choice)
        + 0.01 * float(bool(label_tokens and choice_tokens and label_tokens[0] == choice_tokens[0]))
    )

    if prefix < 0.20:
        score -= 0.05
    if max(len(label), len(choice)) <= 32 and lcs < 0.55:
        score -= 0.05
    if " " not in label and len(label) <= 8:
        if prefix < 0.35:
            score -= 0.08
        if trigrams < 0.30:
            score -= 0.08

    return max(0.0, min(1.0, score))


class FastFuzzyMatcher:
    def __init__(self, choices: Iterable[str]):
        self.choice_data = []
        self.normalized_choices = []
        self.normalized_map = {}

        for choice in choices:
            if is_missing(choice):
                continue
            original = str(choice).strip()
            if not original:
                continue
            normalized = normalize_label(original)
            self.normalized_choices.append(normalized)
            self.choice_data.append(
                {
                    "original": original,
                    "normalized": normalized,
                    "ngrams": char_ngrams(normalized),
                }
            )
            self.normalized_map.setdefault(normalized, []).append(original)

    def get_best_match(
        self,
        label,
        threshold: float = 0.88,
        limit: int = 5,
        margin: float = 0.04,
    ) -> Optional[str]:
        if is_missing(label) or not self.choice_data:
            return None

        label_normalized = normalize_label(str(label).strip())
        if not label_normalized:
            return None
        if label_normalized in self.normalized_map:
            return min(self.normalized_map[label_normalized], key=len)

        candidates = process.extract(
            label_normalized,
            self.normalized_choices,
            scorer=fuzz.WRatio,
            limit=limit,
        )
        rescored = []
        label_ngrams = char_ngrams(label_normalized)
        for _, _, index in candidates:
            choice = self.choice_data[index]
            rescored.append(
                (
                    choice["original"],
                    score_pair(
                        label_normalized,
                        label_ngrams,
                        choice["normalized"],
                        choice["ngrams"],
                    ),
                )
            )

        if not rescored:
            return None
        rescored.sort(key=lambda item: item[1], reverse=True)
        best_choice, best_score = rescored[0]
        second_score = rescored[1][1] if len(rescored) > 1 else 0.0

        effective_threshold = threshold
        if len(label_normalized) <= 7:
            effective_threshold = max(effective_threshold, 0.94)
        elif " " not in label_normalized and len(label_normalized) < 10:
            effective_threshold = max(effective_threshold, 0.92)

        if best_score < effective_threshold or best_score - second_score < margin:
            return None
        return best_choice


def normalize_code(code) -> str:
    if is_missing(code):
        return ""
    code = re.sub(r"[^0-9.]", "", str(code).strip())
    return re.sub(r"\.+", ".", code).strip(".")


def split_code_parts(code: str) -> list[str]:
    normalized = normalize_code(code)
    return normalized.split(".") if normalized else []


def is_descendant_or_same(parent: str, candidate: str) -> bool:
    parent = normalize_code(parent)
    candidate = normalize_code(candidate)
    return bool(parent and candidate and (candidate == parent or candidate.startswith(parent + ".")))


class EscoCodeMatcher:
    def __init__(self, existing_codes: Iterable[str]):
        self.normalized_to_original = {}
        for code in existing_codes:
            normalized = normalize_code(code)
            if normalized:
                self.normalized_to_original.setdefault(normalized, code)
        self.codes = list(self.normalized_to_original)
        self.available_roots = sorted(
            {
                split_code_parts(code)[0]
                for code in self.codes
                if split_code_parts(code) and split_code_parts(code)[0].isdigit()
            }
        )

    def deepest_descendant(self, query: str) -> Optional[str]:
        query = normalize_code(query)
        descendants = [code for code in self.codes if is_descendant_or_same(query, code)]
        if not descendants:
            return None
        descendants.sort(key=lambda code: (code.count("."), len(code), code), reverse=True)
        return self.normalized_to_original[descendants[0]]

    def closest_parent(self, query: str) -> Optional[str]:
        parts = split_code_parts(query)
        for end in range(len(parts) - 1, 0, -1):
            parent = ".".join(parts[:end])
            if parent in self.normalized_to_original:
                return self.normalized_to_original[parent]
        if parts:
            for end in range(len(parts[0]) - 1, 0, -1):
                parent = parts[0][:end]
                if parent in self.normalized_to_original:
                    return self.normalized_to_original[parent]
        return None

    def nearest_root_descendant(self, query: str) -> Optional[str]:
        parts = split_code_parts(query)
        if not parts or not parts[0].isdigit() or not self.available_roots:
            return None
        query_root = int(parts[0])
        nearest_root = min(
            self.available_roots,
            key=lambda root: (abs(int(root) - query_root), int(root)),
        )
        return self.deepest_descendant(nearest_root)

    def find_closest_code(self, query: str) -> Optional[str]:
        if not normalize_code(query):
            return None
        return (
            self.deepest_descendant(query)
            or self.closest_parent(query)
            or self.nearest_root_descendant(query)
        )


def build_altlabel_lookup(df: pd.DataFrame) -> dict[str, str]:
    lookup = {}
    for alt_label, preferred_label in df[["altLabels", "preferredLabel"]].itertuples(index=False):
        if is_missing(alt_label) or is_missing(preferred_label):
            continue
        lookup.setdefault(str(alt_label).casefold(), preferred_label)
    return lookup


def get_best_alt_match(
    label,
    matcher: FastFuzzyMatcher,
    lookup: dict[str, str],
    threshold: float,
) -> Optional[str]:
    alt_label = matcher.get_best_match(label, threshold=threshold, limit=1, margin=0.0)
    if alt_label is None:
        return None
    return lookup.get(alt_label.casefold())


def find_esco_title_by_code(
    code,
    matcher: EscoCodeMatcher,
    code_to_title: dict[str, str],
) -> Optional[str]:
    closest_code = matcher.find_closest_code(code)
    if closest_code is None:
        return None
    return code_to_title.get(normalize_code(closest_code))

## 4.4. Prepare normalised ESCO lookup tables

Explode newline-separated alternative labels, normalise all matching labels, and build the reusable title, code, occupation, and skill lookups once before the daily loop. The fuzzy threshold is fixed at `0.7`; labels of nine characters or fewer are automatically subject to stricter thresholds.

In [ ]:
FUZZY_THRESHOLD = 0.7

skills_df["preferredLabel"] = skills_df["preferredLabel"].apply(normalize_text)
occupations_df["preferredLabel"] = occupations_df["preferredLabel"].apply(normalize_text)

esco_exploded = occupations_df.copy()
esco_exploded["altLabels"] = esco_exploded["altLabels"].fillna("").str.split("\n")
esco_exploded = esco_exploded.explode("altLabels")
esco_exploded["altLabels"] = esco_exploded["altLabels"].apply(normalize_text)
esco_exploded = esco_exploded[["preferredLabel", "altLabels", "code"]]
esco_exploded = esco_exploded[esco_exploded["altLabels"] != ""]
esco_exploded = esco_exploded.drop_duplicates(subset="altLabels", keep="first")

preferred_matcher = FastFuzzyMatcher(occupations_df["preferredLabel"])
alt_matcher = FastFuzzyMatcher(esco_exploded["altLabels"])
code_matcher = EscoCodeMatcher(occupations_df["code"])
alt_lookup = build_altlabel_lookup(esco_exploded)

code_to_title = {}
for code_value, preferred_label in occupations_df[["code", "preferredLabel"]].itertuples(index=False):
    normalized_code = normalize_code(code_value)
    if normalized_code:
        code_to_title.setdefault(normalized_code, preferred_label)

occ_lookup = (
    occupations_df.rename(
        columns={
            "preferredLabel": "esco_title",
            "conceptUri": "esco_id",
            "code": "esco_code",
        }
    )[["esco_title", "esco_id", "esco_code"]]
    .drop_duplicates(subset="esco_title")
)

skills_by_occ = (
    occupation_skills_df.merge(
        skills_df[["conceptUri", "preferredLabel"]],
        left_on="skillUri",
        right_on="conceptUri",
        how="left",
    )
    .groupby("occupationUri")["preferredLabel"]
    .apply(lambda labels: json.dumps(labels.dropna().tolist()))
)

skill_label_map = skills_df.set_index("conceptUri")["preferredLabel"]


def format_labels(labels: pd.Series) -> str:
    values = [label for label in labels.dropna().tolist() if label != ""]
    return "[" + ", ".join(f"'{label}'" for label in values) + "]"


output_columns = [
    "id", "title", "description", "min_salary", "max_salary", "currency",
    "salary_rate", "date_created", "date_expired", "clean_title",
    "clean_desc", "title_lang", "desc_lang", "skill_ids", "skill_labels",
    "job_type", "job_type_score", "classified_code", "classified_title",
    "skill_labels_en", "classified_title_clean", "extract_type",
    "esco_title", "esco_skills", "esco_id", "esco_code",
]

## 4.5. Match occupations and add ESCO skills

For every completed Stage 3 daily file, the notebook applies five matching steps:

1. Exact ESCO preferred label.
2. Exact ESCO alternative label.
3. Fuzzy preferred label.
4. Fuzzy alternative label.
5. ESCO-code fallback.

Unresolved records remain in the output with missing ESCO enrichment fields. Stage 4 outputs remain Pickle files because the Stage 5 rejoin notebooks read `ua-YYYY-MM-DD.pkl` files.

In [ ]:
for process_index, process_row in process_df.iterrows():
    if process_row["result_status"] != "complete":
        continue

    started_at = time.time()
    output_path = os.path.join(
        g.Config.STAGE4_OUTPUT_PATH,
        process_row["input_file"] + ".pkl",
    )
    if os.path.exists(output_path):
        print(f"{process_index} ----- File skipped for {process_row['input_file']}")
        continue

    stage3_data = pd.read_pickle(process_row["result_path"]).reset_index(drop=True)
    stage3_data = stage3_data.drop_duplicates(subset="id", keep="first")
    total_rows = stage3_data.shape[0]

    stage3_data["esco_title"] = (
        stage3_data["esco_title"]
        .replace(["None", "N/A", "NA"], "")
        .fillna("")
    )
    stage3_data = stage3_data.rename(
        columns={"esco_code": "classified_code", "esco_title": "classified_title"}
    )
    stage3_data["classified_title_clean"] = stage3_data["classified_title"].apply(normalize_text)
    stage3_data["extract_type"] = None

    print(f"{process_index} ----- Working on {process_row['input_file']}, lines: {total_rows}")

    preferred_labels = occupations_df[["preferredLabel"]].drop_duplicates()
    exact_preferred = stage3_data[["id", "classified_title_clean"]].merge(
        preferred_labels,
        left_on="classified_title_clean",
        right_on="preferredLabel",
        how="left",
    )
    stage3_data = stage3_data.merge(
        exact_preferred[["id", "preferredLabel"]],
        on="id",
        how="left",
    ).rename(columns={"preferredLabel": "esco_title"})

    ready_data = stage3_data[stage3_data["esco_title"].notna()].copy()
    ready_data["extract_type"] = "preferredLabel"
    not_ready_data = stage3_data[stage3_data["esco_title"].isna()].copy()
    not_ready_data["classified_title_clean"] = not_ready_data["classified_title_clean"].replace("", pd.NA)
    not_ready_data["classified_code"] = not_ready_data["classified_code"].replace(
        ["", "N/A", "NA"], pd.NA
    )
    not_ready_data = not_ready_data.drop(columns="esco_title")
    print(f"STAGE 1 -- matched {ready_data.shape[0]} of {total_rows}")

    if not not_ready_data.empty:
        exact_alt = not_ready_data[["id", "classified_title_clean"]].merge(
            esco_exploded[["altLabels", "preferredLabel"]],
            left_on="classified_title_clean",
            right_on="altLabels",
            how="left",
        )
        not_ready_data = not_ready_data.merge(
            exact_alt[["id", "preferredLabel"]],
            on="id",
            how="left",
        ).rename(columns={"preferredLabel": "esco_title"})
        step_ready = not_ready_data[not_ready_data["esco_title"].notna()].copy()
        step_ready["extract_type"] = "altLabels"
        ready_data = pd.concat([ready_data, step_ready], ignore_index=True)
        not_ready_data = not_ready_data[not_ready_data["esco_title"].isna()].drop(
            columns="esco_title"
        )
        print(f"STAGE 2 -- matched {ready_data.shape[0]} of {total_rows}")

    if not not_ready_data.empty:
        not_ready_data["esco_title"] = not_ready_data["classified_title_clean"].apply(
            lambda value: preferred_matcher.get_best_match(
                value,
                threshold=FUZZY_THRESHOLD,
            )
        )
        step_ready = not_ready_data[not_ready_data["esco_title"].notna()].copy()
        step_ready["extract_type"] = "preferredLabel_fuzzy"
        ready_data = pd.concat([ready_data, step_ready], ignore_index=True)
        not_ready_data = not_ready_data[not_ready_data["esco_title"].isna()].drop(
            columns="esco_title"
        )
        print(f"STAGE 3 -- matched {ready_data.shape[0]} of {total_rows}")

    if not not_ready_data.empty:
        not_ready_data["esco_title"] = not_ready_data["classified_title_clean"].apply(
            lambda value: get_best_alt_match(
                value,
                alt_matcher,
                alt_lookup,
                FUZZY_THRESHOLD,
            )
        )
        step_ready = not_ready_data[not_ready_data["esco_title"].notna()].copy()
        step_ready["extract_type"] = "altLabels_fuzzy"
        ready_data = pd.concat([ready_data, step_ready], ignore_index=True)
        not_ready_data = not_ready_data[not_ready_data["esco_title"].isna()].drop(
            columns="esco_title"
        )
        print(f"STAGE 4 -- matched {ready_data.shape[0]} of {total_rows}")

    if not not_ready_data.empty:
        not_ready_data["esco_title"] = not_ready_data["classified_code"].apply(
            lambda value: find_esco_title_by_code(value, code_matcher, code_to_title)
        )
        step_ready = not_ready_data[not_ready_data["esco_title"].notna()].copy()
        step_ready["extract_type"] = "esco_code_similarity"
        ready_data = pd.concat([ready_data, step_ready], ignore_index=True)
        not_ready_data = not_ready_data[not_ready_data["esco_title"].isna()].copy()
        print(f"STAGE 5 -- matched {ready_data.shape[0]} of {total_rows}")

    ready_data = ready_data.merge(occ_lookup, on="esco_title", how="left")
    ready_data["esco_skills"] = ready_data["esco_id"].map(skills_by_occ).fillna("[]")
    ready_data = pd.concat([ready_data, not_ready_data], ignore_index=True)

    exploded_skill_ids = (
        ready_data["skill_ids"]
        .fillna("")
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )
    matched_skill_labels = exploded_skill_ids.map(skill_label_map)
    skill_labels_en = matched_skill_labels.groupby(level=0).apply(format_labels)
    ready_data["skill_labels_en"] = "[]"
    ready_data.loc[skill_labels_en.index, "skill_labels_en"] = skill_labels_en

    ready_data = ready_data.reindex(columns=output_columns).reset_index(drop=True)
    ready_data.to_pickle(output_path)

    elapsed = time.time() - started_at
    print(
        f"Execution time: {elapsed:.6f} s. "
        f"Done: {ready_data.shape[0]} / missed: {not_ready_data.shape[0]} / total: {total_rows}"
    )

print(time.ctime())

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.